In [ ]:
!git clone --branch v2 https://github.com/Jay-Huy/EBM-Research
%cd EBM-Research
!pip install pyyaml wandb
!wandb login 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

In [ ]:
MODE = "et" # "et" or "forced" # default = "et"
STEP_SIZE = 0.5
CONFIG = "config/sudoku_rrn.yaml"

EPOCHS = 40
BATCH_SIZE = 16

TIE_MODE = "hard" # "orbit" or "hard" | default = "orbit"
GRAD_NORM = "true" # "false" or "true" | default = "false"
MU_PENALTY = 1 # 0.5, 1, 10, 100 | default = 0 (off)
MAX_GRADIENT_STEPS = 16

In [ ]:
!python -c "from data import get_dataset; get_dataset('sudoku_rrn', 'train')"

downloading https://www.dropbox.com/s/rp3hbjs91xiqdgc/sudoku-hard.zip?dl=1 -> data/sudoku/raw/sudoku-hard.zip
extracted to data/sudoku/raw/sudoku-hard


In [ ]:
# Train

import subprocess, time
from pathlib import Path
HEADS = [1, 8]                                    # head 1 -> cuda:0, head 8 -> cuda:1

RUN = (f"results/{Path(CONFIG).stem}-{MODE}-unroll-{{}}h" + ("-hard" if TIE_MODE == "hard" else "")
       + (f"-mu{MU_PENALTY:g}" if MU_PENALTY else ""))

start = time.time()
cmd = (f"python train.py --config {CONFIG} --set model.mode={MODE} model.step_size={STEP_SIZE} model.max_gradient_steps={MAX_GRADIENT_STEPS} "
       f"model.tie_mode={TIE_MODE} train.clip_grad={GRAD_NORM} train.epochs={EPOCHS} data.batch_size={BATCH_SIZE} "
       f"train.mu_penalty={MU_PENALTY} model.n_heads={{}} device=cuda:{{}}")
procs = [subprocess.Popen(cmd.format(h, i), shell=True, stdout=open(f"{MODE}_{h}h.out", "w"), stderr=subprocess.STDOUT)
         for i, h in enumerate(HEADS)]
for p in procs:
    p.wait()
end = time.time()
print(f"train: {(end - start) / 60:.1f} min")

In [ ]:
# Inference

start = time.time()
cmd = f"python inference.py --run {RUN} --split test_id --device cuda:{{}}"
procs = [subprocess.Popen(cmd.format(h, i), shell=True, stdout=open(f"{MODE}_{h}h_inference.out", "w"), stderr=subprocess.STDOUT)
         for i, h in enumerate(HEADS)]
for p in procs:
    p.wait()
end = time.time()
print(f"inference: {(end - start) / 60:.1f} min")

inference: 1.9 min


In [ ]:
# Eval

for h in HEADS:
    run = RUN.format(h)
    !python eval.py --inference {run}/inference/test_id-best

setup
  task        sudoku_rrn
  mode        et
  heads       1
  training    unroll, 16 steps (max_gradient_steps)
  split       test_id (1000 samples)
  checkpoint  best_checkpoint.pt, epoch 1
  Z0          1 (seeds [0])
results
  step 16 (max_gradient_steps)  board_acc 0.0000   cell_acc 14.1067
  fixed point                   board_acc 0.0000   cell_acc 12.9420
  converged (local dist < 1e-05): 20.4000 %, median steps 176.0000
written: results/sudoku_rrn-et-unroll-1h-hard/metrics/test_id-best.txt
setup
  task        sudoku_rrn
  mode        et
  heads       8
  training    unroll, 16 steps (max_gradient_steps)
  split       test_id (1000 samples)
  checkpoint  best_checkpoint.pt, epoch 1
  Z0          1 (seeds [0])
results
  step 16 (max_gradient_steps)  board_acc 0.0000   cell_acc 14.1139
  fixed point                   board_acc 0.0000   cell_acc 13.9498
  converged (local dist < 1e-05): 82.9000 %, median steps 114.0000
written: results/sudoku_rrn-et-unroll-8h-hard/metrics/test_id